# 01 — Legal Inventory Exploration
## Mexico Equality & Cyber Literacy Atlas · LGBTIQ+ Rights Compliance Pipeline

This notebook **inventories** state-level laws from [Orden Jurídico Nacional](http://www.ordenjuridico.gob.mx) for all 32 Mexican states.

**Goals:**
- Discover which ordenamientos (laws/regulations) are publicly available per state.
- Flag documents likely relevant to the project's compliance markers:
  - **L1** Explicit non-discrimination protection based on sexual orientation / gender identity
  - **L2** Legal gender-identity recognition
  - **L3** Same-sex union or marriage rights
  - **C1** Hate-crime / aggravated penalty provisions
  - **C2** Anti-discrimination enforcement mechanism
  - **C3** Healthcare access without discrimination

**Scope:** URL inventory only — no PDF downloads, no NLP yet.

---
## 1 · Setup & Imports

Standard library only — `requests` for HTTP, `BeautifulSoup` for HTML parsing,
`pandas` for tabular results, and `time` for polite crawl delays.

In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import unicodedata
import re

# ── HTTP session ──────────────────────────────────────────────────────────────
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (compatible; LegalInventoryBot/1.0; "
        "academic research - LGBTIQ+ rights compliance atlas)"
    )
})

# Use plain HTTP -- the site serves both HTTP and HTTPS but its SSL certificate
# cannot be verified by Python's default CA bundle on Windows, so HTTPS raises
# SSLCertVerificationError.  HTTP is sufficient for a read-only public inventory.
BASE_URL    = "http://www.ordenjuridico.gob.mx"
LISTING_URL = BASE_URL + "/despliegaedo2.php?ordenar=&edo={code}&catTipo="
FICHA_URL   = BASE_URL + "/fichaOrdenamiento.php?idArchivo={doc_id}&ambito=estatal"

CRAWL_DELAY = 2.0   # seconds between requests -- be polite

print("Imports OK -- pandas", pd.__version__)

Imports OK -- pandas 2.2.3


---
## 2 · State Catalog

Maps each of Mexico's 32 federal entities to the `edo` code used by
`ordenjuridico.gob.mx` in its `/estatal.php?edo=<code>` URLs.

> **Source for codes:** verified against the site's own navigation links.
> CDMX appears as `DF` in many older URLs but the site also accepts `CDMX`;
> both are included as a fallback tuple where uncertain.

In [7]:
# State name -> numeric INEGI code used by ordenjuridico.gob.mx
# These are the standard two-digit INEGI claves geoestadisticas (01-32).
# The site's despliegaedo2.php accepts them as the `edo` parameter.
STATE_CATALOG: dict[str, str] = {
    "Aguascalientes":      "01",
    "Baja California":     "02",
    "Baja California Sur": "03",
    "Campeche":            "04",
    "Coahuila":            "05",
    "Colima":              "06",
    "Chiapas":             "07",
    "Chihuahua":           "08",
    "Ciudad de Mexico":    "09",
    "Durango":             "10",
    "Guanajuato":          "11",
    "Guerrero":            "12",
    "Hidalgo":             "13",
    "Jalisco":             "14",
    "Estado de Mexico":    "15",
    "Michoacan":           "16",
    "Morelos":             "17",
    "Nayarit":             "18",
    "Nuevo Leon":          "19",
    "Oaxaca":              "20",
    "Puebla":              "21",
    "Queretaro":           "22",
    "Quintana Roo":        "23",
    "San Luis Potosi":     "24",
    "Sinaloa":             "25",
    "Sonora":              "26",
    "Tabasco":             "27",
    "Tamaulipas":          "28",
    "Tlaxcala":            "29",
    "Veracruz":            "30",
    "Yucatan":             "31",
    "Zacatecas":           "32",
}

assert len(STATE_CATALOG) == 32, f"Expected 32 states, got {len(STATE_CATALOG)}"
print(f"Catalog loaded: {len(STATE_CATALOG)} states")
print("URL example:", LISTING_URL.format(code="31"), "(Yucatan)")

Catalog loaded: 32 states
URL example: http://www.ordenjuridico.gob.mx/despliegaedo2.php?ordenar=&edo=31&catTipo= (Yucatan)


---
## 3 · Law Discovery per State

`scrape_state(state_name, code)` fetches the state page from Orden Jurídico
Nacional and returns a list of dicts, one per discovered ordenamiento.

**Scraping strategy:**
1. Fetch `/estatal.php?edo={code}` — the main landing page for the state.
2. Find all `<a>` tags whose `href` points to a `.pdf` or an internal HTML page
   under `/contenido/` or `/federal/` (the site's standard document paths).
3. Resolve relative URLs against the base domain.
4. Classify each link as `PDF` or `HTML`.

Because the site renders some sections via JavaScript, we also look for links
ending with `.php` that carry a document reference, as these are common
ordenamiento viewer URLs on this site.

In [8]:
def _normalize(text: str) -> str:
    """Lower-case, strip accents, collapse whitespace -- for keyword matching."""
    nfkd = unicodedata.normalize("NFKD", text)
    ascii_text = nfkd.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", ascii_text).strip().lower()


# Regex to extract idArchivo from the javascript:void(window.open(...)) href.
# Example href value:
#   javascript:void(window.open("fichaOrdenamiento.php?idArchivo=121407&ambito=estatal",...))
_FICHA_RE = re.compile(r'idArchivo=(\d+)', re.IGNORECASE)


def scrape_state(state_name: str, code: str) -> list[dict]:
    """
    Fetch despliegaedo2.php for the given INEGI numeric *code* and return all
    ordenamiento entries as a list of dicts with keys:
        state, law_title, url, file_type, law_type
    """
    url = LISTING_URL.format(code=code)
    entries: list[dict] = []

    try:
        resp = SESSION.get(url, timeout=30)
        resp.encoding = "iso-8859-1"   # site declares ISO-8859-1
        resp.raise_for_status()
    except requests.RequestException as exc:
        print(f"  [WARN] {state_name} ({code}): request failed -- {exc}")
        return entries

    soup = BeautifulSoup(resp.text, "html.parser")

    # Each law is a <tr> with two <td>s:
    #   td[0]: <a href="javascript:void(window.open('fichaOrdenamiento.php?idArchivo=XXXXX...'))">Title</a>
    #   td[1]: law type text (Ley, Decreto, Reglamento, Acuerdo, ...)
    seen_ids: set[str] = set()

    for tr in soup.find_all("tr"):
        tds = tr.find_all("td")
        if len(tds) < 1:
            continue

        a_tag = tds[0].find("a", href=True)
        if a_tag is None:
            continue

        href = a_tag.get("href", "")
        match = _FICHA_RE.search(href)
        if not match:
            continue                   # not a document link

        doc_id = match.group(1)
        if doc_id in seen_ids:
            continue
        seen_ids.add(doc_id)

        title = a_tag.get_text(separator=" ", strip=True)
        law_type = tds[1].get_text(strip=True) if len(tds) > 1 else ""

        entries.append({
            "state":     state_name,
            "law_title": title,
            "url":       FICHA_URL.format(doc_id=doc_id),
            "file_type": "HTML",       # fichaOrdenamiento pages are HTML viewers
            "law_type":  law_type,
        })

    return entries


# ── Sanity check on Yucatan (code 31) ────────────────────────────────────────
print("Sanity-checking scraper on Yucatan [31]...")
sample = scrape_state("Yucatan", "31")
print(f"  Found {len(sample)} entries")
if sample:
    print("  First entry :", sample[0])
    print("  Last entry  :", sample[-1])
time.sleep(CRAWL_DELAY)

Sanity-checking scraper on Yucatan [31]...
  Found 393 entries
  First entry : {'state': 'Yucatan', 'law_title': 'Acuerdo AAFY 11/2020 por el que se establecen disposiciones administrativas complementarias en materia fiscal para el ejercicio fiscal 2020', 'url': 'http://www.ordenjuridico.gob.mx/fichaOrdenamiento.php?idArchivo=121407&ambito=estatal', 'file_type': 'HTML', 'law_type': 'Acuerdo'}
  Last entry  : {'state': 'Yucatan', 'law_title': 'Reglamento para regular los procesos de selección de candidatos a cargos de Elección Popular y Precampañas Electorales en el Estado de Yucatán', 'url': 'http://www.ordenjuridico.gob.mx/fichaOrdenamiento.php?idArchivo=99008&ambito=estatal', 'file_type': 'HTML', 'law_type': 'Reglamento'}


In [9]:
# ── Full crawl across all 32 states ─────────────────────────────────────────
all_raw: list[dict] = []

for state_name, code in STATE_CATALOG.items():
    print(f"  Fetching {state_name:25s} [{code:>2}]", end=" ... ", flush=True)
    entries = scrape_state(state_name, code)
    all_raw.extend(entries)
    print(f"{len(entries):4d} entries")
    time.sleep(CRAWL_DELAY)

print(f"\nTotal raw entries across all 32 states: {len(all_raw)}")

  Fetching Aguascalientes            [01] ... 1568 entries
  Fetching Baja California           [02] ...  201 entries
  Fetching Baja California Sur       [03] ...  262 entries
  Fetching Campeche                  [04] ...  218 entries
  Fetching Coahuila                  [05] ... 2716 entries
  Fetching Colima                    [06] ... 1086 entries
  Fetching Chiapas                   [07] ...  548 entries
  Fetching Chihuahua                 [08] ... 1507 entries
  Fetching Ciudad de Mexico          [09] ... 5821 entries
  Fetching Durango                   [10] ...  411 entries
  Fetching Guanajuato                [11] ... 1146 entries
  Fetching Guerrero                  [12] ...  337 entries
  Fetching Hidalgo                   [13] ...  955 entries
  Fetching Jalisco                   [14] ... 2156 entries
  Fetching Estado de Mexico          [15] ... 1804 entries
  Fetching Michoacan                 [16] ...  974 entries
  Fetching Morelos                   [17] ...    5 entri

---
## 4 · Relevant Law Filtering

We filter the raw inventory to keep only laws whose **title** contains at least
one keyword that signals relevance to the L1–L3 / C1–C3 compliance markers.

Keyword groups:

| Marker | Keywords |
|--------|----------|
| L1 / C2 | discriminación, igualdad, no discriminación |
| L2 | identidad de género, cambio de nombre, registro civil |
| L3 | matrimonio, sociedad de convivencia, unión civil, concubinato |
| C1 | código penal, violencia, crimen de odio, agravante |
| C2 | derechos humanos, comisión estatal |
| C3 | salud, servicios de salud, atención médica |
| General | familia, orientación sexual, diversidad sexual, lgbtq, lgbtiq |

Matching is done on the **accent-stripped, lower-cased** title so diacritics
don't cause misses.

In [10]:
# Keyword list — normalized (accent-stripped, lower-case) so matching is robust
KEYWORDS: list[str] = [
    # L1 / C2 — non-discrimination & equality
    "discriminacion",
    "no discriminacion",
    "igualdad",
    # L2 — gender identity recognition
    "identidad de genero",
    "cambio de nombre",
    "registro civil",
    # L3 — same-sex unions / marriage
    "matrimonio",
    "sociedad de convivencia",
    "union civil",
    "concubinato",
    # C1 — criminal provisions / hate crimes / violence
    "codigo penal",
    "violencia",
    "crimen de odio",
    "agravante",
    # C2 — human-rights enforcement bodies
    "derechos humanos",
    "comision estatal",
    # C3 — healthcare access
    "salud",
    "servicios de salud",
    "atencion medica",
    # General LGBTIQ+ scope
    "familia",
    "orientacion sexual",
    "diversidad sexual",
    "lgbtq",
    "lgbtiq",
    "genero",
]


def match_keywords(title: str) -> list[str]:
    """Return the subset of KEYWORDS found in the normalized title."""
    norm = _normalize(title)
    return [kw for kw in KEYWORDS if kw in norm]


# Apply filter
relevant_raw: list[dict] = []
for entry in all_raw:
    hits = match_keywords(entry["law_title"])
    if hits:
        relevant_raw.append({**entry, "matched_keywords": hits})

print(f"Raw entries      : {len(all_raw):>5}")
print(f"Relevant entries : {len(relevant_raw):>5}")
print(f"Retention rate   : {len(relevant_raw)/max(len(all_raw),1)*100:.1f}%")

Raw entries      : 30532
Relevant entries :  2639
Retention rate   : 8.6%


---
## 5 · Results as DataFrame

Consolidate the filtered inventory into a tidy `pandas.DataFrame` with columns:

| Column | Description |
|--------|-------------|
| `state` | Full state name |
| `law_title` | Document title as scraped from the anchor text |
| `url` | Absolute URL on ordenjuridico.gob.mx |
| `file_type` | `PDF` or `HTML` |
| `matched_keywords` | Comma-separated list of matched keywords |

In [11]:
df = pd.DataFrame(relevant_raw)

if not df.empty:
    # Serialize keyword list as a readable comma-separated string
    df["matched_keywords"] = df["matched_keywords"].apply(lambda kws: ", ".join(kws))

    # Enforce column order (law_type is bonus metadata, not in the original spec)
    df = df[["state", "law_title", "url", "file_type", "matched_keywords", "law_type"]]
    df = df.reset_index(drop=True)
else:
    df = pd.DataFrame(columns=["state", "law_title", "url", "file_type", "matched_keywords", "law_type"])

print(f"DataFrame shape: {df.shape}")
print(f"\nLaw-type breakdown:\n{df['law_type'].value_counts().to_string()}")
print(f"\nSample rows:")
df.head(10)

DataFrame shape: (2639, 6)

Law-type breakdown:
law_type
Acuerdo                    515
Decreto Legislativo        491
Reglamento                 371
Convenio                   320
Ley                        295
Decreto Administrativo      94
Aviso                       85
Manual                      56
Decreto                     56
Lineamiento                 51
Regla                       50
Código                      47
Programa                    38
Anexo                       21
Nota Aclaratoria            17
Circular                    15
Fe de erratas               15
Estatuto                    12
Disposiciones diversas      12
Reforma                     11
Estado                       9
Protocolo                    7
Convocatoria                 7
Acta                         6
Resolución                   4
Criterio                     3
Procedimiento                3
Informe                      2
Disposición General          2
Declaración                  2
Declaratoria 

,state,law_title,url,file_type,matched_keywords,law_type
0,Aguascalientes,Acta de Sesión de la Comisión Estatal de Desar...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,comision estatal,Acta
1,Aguascalientes,Acta de sesión extraordinaria celebrada el día...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,comision estatal,Acta
2,Aguascalientes,Acta de Sesión Extraordinaria celebrada el día...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,comision estatal,Acta
3,Aguascalientes,Acta de Sesión Ordinaria celebrada el día 15 d...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,comision estatal,Acta
4,Aguascalientes,Acuerdo de Coordinación para garantizar la pre...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,"salud, servicios de salud",Acuerdo
5,Aguascalientes,Acuerdo de Coordinación para la Ejecución del ...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,"salud, servicios de salud",Acuerdo
6,Aguascalientes,"Acuerdo General 001/2020, del Pleno del\r\nIns...",http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,salud,Acuerdo
7,Aguascalientes,"Acuerdo General 002/2020, del Pleno del Instit...",http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,salud,Acuerdo
8,Aguascalientes,Acuerdo General Número 14/05/2020 por el que s...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,salud,Acuerdo
9,Aguascalientes,Acuerdo General número 19/03/2020 por el que s...,http://www.ordenjuridico.gob.mx/fichaOrdenamie...,HTML,salud,Acuerdo


---
## 6 · Coverage Report

Two outputs:
1. **Per-state count** — how many relevant laws were found for each state.
2. **Zero-coverage states** — states with no hits that will require a manual
   document search (e.g., Google Drive folder with PDF fallbacks).

> States with zero automatic matches are **not** necessarily non-compliant —
> the law may exist under a non-obvious title or only inside a PDF that wasn't
> indexed as a named link on the Orden Jurídico page.

In [12]:
# ── Per-state count ──────────────────────────────────────────────────────────
all_states = list(STATE_CATALOG.keys())

counts = (
    df.groupby("state")
    .size()
    .reindex(all_states, fill_value=0)
    .sort_values(ascending=False)
    .rename("relevant_laws")
)

print("=" * 50)
print(f"{'State':<28} {'Relevant laws':>13}")
print("=" * 50)
for state, n in counts.items():
    flag = "  ← needs manual fallback" if n == 0 else ""
    print(f"  {state:<26} {n:>6}{flag}")
print("=" * 50)
print(f"  {'TOTAL':<26} {counts.sum():>6}")
print(f"  {'States with ≥1 match':<26} {(counts > 0).sum():>6} / 32")

State                        Relevant laws
  Ciudad de Mexico              442
  Coahuila                      258
  Jalisco                       204
  Aguascalientes                193
  Chihuahua                     161
  Estado de Mexico              161
  Veracruz                      158
  Colima                        118
  Hidalgo                        91
  Sinaloa                        86
  Michoacan                      77
  Tamaulipas                     66
  Guanajuato                     60
  Yucatan                        55
  Oaxaca                         49
  Nayarit                        48
  Quintana Roo                   43
  Puebla                         41
  Zacatecas                      38
  Queretaro                      38
  San Luis Potosi                37
  Chiapas                        36
  Durango                        34
  Tabasco                        31
  Guerrero                       29
  Baja California Sur            25
  Campeche           

In [13]:
# ── Zero-coverage states (need Drive fallback) ───────────────────────────────
zero_coverage = counts[counts == 0].index.tolist()

if zero_coverage:
    print(f"\n{len(zero_coverage)} state(s) with ZERO automatic matches — manual Drive fallback required:\n")
    for s in zero_coverage:
        print(f"  • {s}")
else:
    print("\nAll 32 states have at least one automatically discovered relevant law.")

print("\n── Keyword hit distribution ──────────────────────────────────────────────")
if not df.empty:
    from collections import Counter
    all_kws = Counter(
        kw.strip()
        for kw_str in df["matched_keywords"]
        for kw in kw_str.split(",")
    )
    for kw, cnt in all_kws.most_common():
        print(f"  {kw:<35} {cnt:>4} documents")


1 state(s) with ZERO automatic matches — manual Drive fallback required:

  • Morelos

── Keyword hit distribution ──────────────────────────────────────────────
  salud                               1118 documents
  familia                              441 documents
  codigo penal                         313 documents
  violencia                            257 documents
  comision estatal                     228 documents
  derechos humanos                     224 documents
  servicios de salud                   185 documents
  igualdad                             100 documents
  registro civil                        78 documents
  discriminacion                        74 documents
  genero                                55 documents
  atencion medica                       14 documents
  no discriminacion                      9 documents
  matrimonio                             5 documents
  orientacion sexual                     2 documents
  cambio de nombre                       1

In [14]:
# ── Export inventory to CSV (optional — uncomment to save) ──────────────────
# Saving the raw inventory means future runs don't need to re-crawl the site.

# df.to_csv("data/legal_inventory.csv", index=False, encoding="utf-8-sig")
# print("Saved → data/legal_inventory.csv")

# Also export the full raw list (pre-filter) for reference:
# pd.DataFrame(all_raw).to_csv("data/legal_inventory_raw.csv", index=False, encoding="utf-8-sig")
# print("Saved → data/legal_inventory_raw.csv")

print("Export cells are commented out. Uncomment and run to persist results.")
print(f"\nInventory summary:")
print(f"  Total raw entries  : {len(all_raw)}")
print(f"  Filtered entries   : {len(df)}")
print(f"  States covered     : {df['state'].nunique()} / 32")
print(f"  States needing fallback: {len(zero_coverage)}")

Export cells are commented out. Uncomment and run to persist results.

Inventory summary:
  Total raw entries  : 30532
  Filtered entries   : 2639
  States covered     : 31 / 32
  States needing fallback: 1


---
## Next Steps

| Step | Notebook | Description |
|------|----------|-------------|
| Download | `02_document_download.ipynb` | Fetch PDFs and HTML for entries in `df` |
| Parse | `03_text_extraction.ipynb` | Extract clean text from PDFs (pdfplumber / pdfminer) |
| Index | `04_rag_indexing.ipynb` | Chunk + embed with LlamaIndex |
| Evaluate | `05_compliance_scoring.ipynb` | Score each state against L1–L3 / C1–C3 |

States in `zero_coverage` will need PDFs sourced manually (e.g., government
portals, Google Drive archive) before they can enter the pipeline.